In [ ]:
import pandas as pd
import gdown
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from torchsummary import summary
from MLP import *

In [ ]:
ID_FILE = '1rQAbLeupIKA9VU3NoZldJO1IPXbCx7ew'

In [ ]:
FILE_name = 'Auto_MPG_data.csv'

In [ ]:
Folder_name = 'data/'

In [ ]:
os.makedirs(Folder_name, exist_ok=True)

In [ ]:
# Tải dl từ trên drive về bằng lệnh gdown
gdown.download(id=ID_FILE, output = Folder_name + FILE_name)

# Đọc và hiển thị thông tin cơ bản của bộ DL

In [ ]:
df = pd.read_csv(Folder_name + FILE_name)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Lấy ra đặc trưng và nhãn
X, y = df.iloc[:,1:], df.iloc[:,0]
X

In [ ]:
X = X.to_numpy()
y = y.to_numpy()

In [ ]:
# Chia tập DL thành trainset và testset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42, shuffle=True)

In [ ]:
# Chuẩn hóa DL 
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
X_train

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype = torch.float32)

In [ ]:
class MLPRegressor(BaseMLP):
    def __init__(self):
        super().__init__()
    def predict(self, X):
        return self.forward(X)
    def get_accuracy(self, logits, y):
        y = y.reshape(-1,1)
        ssres = torch.sum((y-logits)**2)
        sstol = torch.sum((y - torch.mean(y))**2)
        return 1 - (ssres/sstol)
    def compute_loss(self, logits, y):
        y = y.reshape(-1,1)
        return self.criterion(logits,y)

In [ ]:
# Khởi tạo mô hình và huấn luyện (validation_split = 0.1)
input_dim = X_train.shape[1]

model = MLPRegressor()
model.Add_layer(nn.Linear(input_dim, 8))
model.Add_layer(nn.GELU())
model.Add_layer(nn.Linear(8, 5))
model.Add_layer(nn.GELU())
model.Add_layer(nn.Linear(5, 1))
summary(model.model, input_size=(9,), batch_size=32, device='cpu')

In [ ]:
# Huấn luyện mô hình 
model.fit(
    X=X_train,
    y=y_train,
    lr=0.01,
    n_epochs=100,
    batch_size=16,
    optimizer='adam',
    criterion='mse',
    validation_split=0.1,
    verbose=2,
)

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(model.Losses) + 1)

plt.figure(figsize=(12, 4))

# Loss
plt.subplot(1, 2, 1)
plt.plot(epochs, model.Losses, label='Train loss')
if getattr(model, 'Val_Losses', None):
    plt.plot(epochs, model.Val_Losses, label='Val loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.legend()

# Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs, model.Accuracies, label='Train accuracy')
if getattr(model, 'Val_Accuracies', None):
    plt.plot(epochs, model.Val_Accuracies, label='Val accuracy')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# Đánh giá mô hình trên tập test
model.model.eval()
with torch.no_grad():
    y_pred = model.predict(X_test.to(model.device)).cpu()
    y_true = y_test.reshape(-1, 1).cpu()

    ssres = torch.sum((y_true - y_pred) ** 2)
    sstol = torch.sum((y_true - torch.mean(y_true)) ** 2)
    r2_test = (1 - ssres / sstol).item()

    mse_test = torch.mean((y_true - y_pred) ** 2).item()
    mae_test = torch.mean(torch.abs(y_true - y_pred)).item()

print(f"Test R2 (theo get_accuracy): {r2_test:.4f}")
print(f"Test MSE: {mse_test:.4f}")
print(f"Test MAE: {mae_test:.4f}")